# Capstone Task
* Create a synthetic data generator using open source models.
* It should generate data on the basis of user request.
* Then build a Gradio interface on top of this.




In [1]:
# Let us decompose the problem first, then we go step by step.

# We are looking for text generation in a given format. The user must provide: a detail of what is needed, example for accurate results, number of rows for which this data is needed.
# We need to have a system prompt, a chat model loaded in the GPU, and ask user for the input. We output each row one by one and stream the results. We do a hardstop when the result output count matches the required data count. We limit our selves to excel like data or tabular data only for this one.
# Once, this is acheived, we work on decomposition of the Gradio piece

In [1]:
# now one decision that we have to take here is to understand what is better? Using pipelines or using apply_chat_template and loading the model manually. Pipeline is less code. But let us understand which one is more efficient. it is more about control on model weights and fine tuning, but that is not our concern. So, we stick with pipeline.

# Now, what all libraries are needed: huggingface_hub/login for login, google.colab for secrets, from transformers: accelerate for auto memmory managent, bitsandbytesconfig for quant, pipeline for textgeneration, , from IPython: display and markdown, and finally pytorch for describing parameter loading size format.

# we start by installing
!pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 59.4 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
from google.colab import userdata
from transformers import pipeline, BitsAndBytesConfig
from IPython.display import display, Markdown
import torch

In [3]:
# now, we login and check if we are connected to the GPU or not
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

In [4]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if "CUDA" in gpu_info:
  print("Connected to GPU")
  print("\n", gpu_info)
else:
  print("GPU not connected")

Connected to GPU

 Thu Sep  3 14:52:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------------------

In [5]:
# next up is the model that we will be using Qwen/Qwen3-8B, which is the most downloaded text generation model.
MODEL = "Qwen/Qwen3-8B"

# since, this is a 8B model, we also need to quantize it.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# @title
# # now, lets draft a simple user message, to check if model is working fine or not, we create a simple message, create pipeline, then test the model.
# test_messages = [
#     {"role": "system", "content": "you are a helpful assistant"},
#     {"role": "user", "content": "tell me a light hearted joke about LLMs"}
# ]

# generate_text = pipeline(
#     "text-generation",
#     model=MODEL,
#     device_map="auto",
#     dtype=torch.bfloat16,
#     model_kwargs={"quantization_config": quant_config}
# )

In [ ]:
# @title
# # also, lets check if quant_config is working fine or not, for this we run:
# allocated_bytes = torch.cuda.memory_allocated()
# allocated_mb = allocated_bytes / (1024 * 1024)
# print(allocated_mb)

In [ ]:
# @title
# # quant is working fine, model size is in excess of 16gbs, we have it stored for 6gbs. now, lets run the prompt and get the output, we then parse it, to get the actual text generated by the model.
# result = generate_text(test_messages, max_new_tokens=2000)

In [ ]:
# @title
# print(result)

In [ ]:
# @title
# # output is a list, so we have result[0], now, we have dict, with key, 'generated_text', so next level is result[0]["generated_text"], this again gives a list, on the basis of our input, we got assistant output, so, we have the 3rd item or 2nd index in the list as the main output. In this, again we have a dict and main key in this is 'content'. so finally we print:
# print(result[0]["generated_text"][2]['content'])

In [ ]:
# @title
# # we knew that this is a thinking model, but do not want to show the thinking part, we only want to show the final output
# # let us first store this in a var, so that we can operate on it(there is another way to prevent from thinking, by applying chat template and passing in thinking as false)
# output = result[0]["generated_text"][2]['content']
# # once we have this stored, we can slice the sentence and store only the final output, for this, we can use .find

In [ ]:
# @title
# think_end_index = output.find("</think>")
# # now, point to be noted is that we will get the index of < in </think>", so, for getting the final output, we need to add 10(when accounting for line breaks also) to the index and then finally slice it. Lets try it out.
# think_end_index += 10
# final_output = output[think_end_index:]
# display(Markdown(final_output))


In [ ]:
# @title
# # Now, this is a bit of a work around, and more robust way is to work on applying the chat template and then getting the final output, but we stick to this for now.
# # let's test this again if this work around is working or not
# test_2_messages = [
#     {"role": "system", "content": "you are a helpful assistant"},
#     {"role": "user", "content": "what all can you do for me?"}
# ]
# result = generate_text(test_2_messages, max_new_tokens=2000)
# output = result[0]["generated_text"][2]['content']
# think_end_index = output.find("</think>")
# think_end_index += 10
# final_output = output[think_end_index:]
# display(Markdown(final_output))


In [ ]:
# @title
# # the model is working fine, we can now proceed ahead with building our system prompt.
# # for what we did in terms of trimming, here's an improvement that we can do, since +10 can be changed by the LLM, so we simply take "</think>", split the output at this, this will give us a list of items before and after </think>, we then take out the second part of the list, and then trim it.
# list_output = output.split("</think>")
# print(list_output)

In [ ]:
# @title
# # yesterday we built a workaround on the think part using pipelines, however, in case a model behaves differently or changes the keyword from think to something else, our code will fail, therefore, we build the model using pipeline and tokenizer using the correct way, commenting out the code that was written yesterday. We start from defining the quant, also we keep one step in one cell, so that if we need to rerun that step later for a further down the line step, we do not end up running multiple steps.
# # Now we have the login in place, all libs in place, all installations in place, with quant_config set.
# # also, since we will be using apply_chat_template and also creating a tokenizer with pipeline to enable no thinking, we would need to import these also, however, online there is a softer way to achieve this, lets try this out before we go down that path, by passing in /no_think in the prompt. Plus lets also use skip_special_tokens=True, while creating the pipeline.

# no_think_system_prompt = "/no_think You are a helpful assistant. You always respond in markdown."
# messages = [
#     {"role": "system", "content": no_think_system_prompt},
#     {"role": "user", "content": "Tell me a joke about future of LLMs/AI"}
# ]

In [ ]:
# @title
# # set up pipeline
# generate_text = pipeline(
#     "text-generation",
#     model=MODEL,
#     dtype=torch.bfloat16,
#     device_map='auto',
#     skip_special_tokens=True,
#     model_kwargs={"quantization_config": quant_config}
# )

In [ ]:
# @title
# # check system RAM consumption:
# ram_bytes = torch.cuda.device_memory_used()
# print(ram_bytes/1e6)

In [ ]:
# result = generate_text(messages, skip_special_tokens=True,)

In [ ]:
# result

In [ ]:
# okay, so no think works fine, but lets move to the completely loaded model using tokenizer and autocausallm, to suppress this and check if we get a cleaner response
# from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# system_prompt = "You are a helpful assistant. You always respond in markdown."
# messages = [
#     {"role": "system", "content": system_prompt},
#     {"role": "user", "content": "Tell me a joke about future of LLMs/AI"}
# ]

In [ ]:
# model_tokenizer = AutoTokenizer.from_pretrained(MODEL)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [ ]:
# model_inputs = model_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, enable_thinking=False, return_tensors="pt").to("cuda")

In [ ]:
# model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", quantization_config=quant_config).to("cuda")

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# # check storage being used
# ram_used = torch.cuda.device_memory_used()
# ram_used/1e6

6934.102016

In [ ]:
# output = model.generate(**model_inputs, max_new_tokens=300)

In [ ]:
# output

tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
           1446,   2677,   5889,    304,  50494,     13, 151645,    198, 151644,
            872,    198,  40451,    752,    264,  21646,    911,   3853,    315,
            444,  10994,     82,  10360,     40, 151645,    198, 151644,  77091,
            198, 151667,    271, 151668,    271,   8420,    594,    264,  21646,
            911,    279,   3853,    315,    444,  10994,     82,    323,  15235,
           1447,  44364,  10234,   1521,    279,  15235,    633,  28926,    311,
            279,   1909,    315,    279,   2813,   1939,  17949,    432,    572,
            279,   3070,   3562,  59236,    334,   9364,   1959,    432,   1410,
          17595,    279,   3853,    315,    279,   2562,    353,    437,      9,
            279,   9104,   2219,  44364,  10061,    752,   1414,    421,    498,
           4172,   1075,    264,    803,  10916,    476,  69846,   1896,      0,
          26525,    226, 151

In [ ]:
# model_tokenizer.decode(output)

["<|im_start|>system\nYou are a helpful assistant. You always respond in markdown.<|im_end|>\n<|im_start|>user\nTell me a joke about future of LLMs/AI<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nHere's a joke about the future of LLMs and AI:\n\n---\n\nWhy did the AI get promoted to the top of the company?\n\nBecause it was the **most predictive** employee — it could forecast the future of the business *and* the weather!\n\n---\n\nLet me know if you'd like a more technical or humorous take! 😄<|im_end|>"]

In [ ]:
#so the format is same even with model loaded manually. so we stick to pipelines for this, and use the simple prompt function to run the model. Now, since, all of this is working, we move ahead with creation of one bigger function, in which we can pass on the message and get the respone, then we go step by step and work on creation of the actual task.

#

def get_response(messages):
  generate_text = pipeline(
      "text-generation",
      model=MODEL,
      device_map='auto',
      dtype=torch.bfloat16,
      model_kwargs={"quantization_config": quant_config}
  )
  model_output = generate_text(messages)
  assistant_response = model_output[0]["generated_text"][2]['content']
  clean_response = assistant_response.split('</think>')[-1].strip()
  return clean_response

In [ ]:
no_think_system_prompt = "/no_think You are a helpful assistant. You always respond in markdown."

messages = [
    {"role": "system", "content": no_think_system_prompt},
    {"role": "user", "content": "Tell me a joke about future of LLMs/AI"}
]

In [ ]:
get_response(messages)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'Here\'s a joke about the future of LLMs and AI:\n\n---\n\nWhy did the AI apply for a job at the time machine?\n\nBecause it wanted to be *predictive* about the future — and it heard the job description said, "We\'re looking for someone who can *forecast* trends, *generate* ideas, and *learn* from the past to *optimize* the present."\n\nThe AI said, "I can do all that — and I also know how to *debug* the timeline."\n\n---\n\nLet me know if you want a more technical or philosophical take! 😄'

In [ ]:
# Now, after running this model multiple times, I can see that weights are being loaded every time. So, we need to create a function, where, model loads once and then we continue generating the content.
# let us first try with checking with pipe.model and look at the results that we get, for that lets restart the runtime, and load model outside the function again, since, we can not access it from outside of the function and running it again will load the model.

In [ ]:
generate_text = pipeline(
    "text-generation",
    model=MODEL,
    device_map='auto',
    dtype=torch.bfloat16,
    model_kwargs={"quantization_config": quant_config}
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [ ]:
print(generate_text.model)

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear4bit(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06

In [28]:
# okay, when we run this command, we get the above info, same as we were getting by doing only model, when we ran this command after manually loading the model, now check what do we get when the model is not loaded and we run this command, for this, however, we need to first delete the model and clean the memory
import gc

In [ ]:
del generate_text.model
torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()

In [ ]:
print(generate_text.model)

AttributeError: 'TextGenerationPipeline' object has no attribute 'model'

In [ ]:
# in this case we get an attribute error, because generate_text is a variable, however, if we have a new variable all together, such as pipe, then
print(pipe.model)

NameError: name 'pipe' is not defined

In [ ]:
# now, we get a name error, since pipe is not defined. Here's what we need to do: we simply pass on a user input, that gets added to the systeminput, becomes a complete inputmesage, this we pass on in the function, now, either we check in this function itself, if model exists, if it exists, we pass on the message, if it does not, we first load the model and the pass on the message. Also, I have checked for torch.cuda and here, it is only about the hardware, will tell us what is the memory and all, but not the name of the model. so, we use pipe.model path only. However, for phase 1, we can simple load the model, then pass in the information to get the output, lets work on this for now.

pipe = pipeline(
    "text-generation",
    model=MODEL,
    device_map='auto',
    dtype=torch.bfloat16,
    model_kwargs={"quantization_config": quant_config}
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [ ]:
# we first test using a simple system and user prompt and then we modify this at each step
system_prompt = """/no_think You are a synthetic data generator.
Generate only clean, comma-separated values (CSV).
Do not include explanations, markdown formatting (no backticks), or conversational text.
Output the CSV header row first, followed immediately by the requested number of data rows."""

user_prompt = """Generate 5 realistic customer records.

Schema:
- customer_id: integer starting at 1001
- full_name: string (realistic first and last name)
- age: integer between 18 and 75
- country: categorical (US, CA, UK, or DE)
- signup_date: date in YYYY-MM-DD format (between 2023-01-01 and 2024-01-01)
- is_active: boolean (true or false)"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
model_output = pipe(messages)
assistant_response = model_output[0]["generated_text"][2]['content']
clean_response = assistant_response.split('</think>')[-1].strip()
print(clean_response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


customer_id,full_name,age,country,signup_date,is_active
1001,John Doe,34,US,2023-05-12,true
1002,Jane Smith,29,CA,2023-08-22,true
1003,Michael Johnson,45,UK,2023-03-10,false
1004,Emily Davis,22,DE,2023-11-05,true
1005,William Brown,60,US,2023-12-18,false


In [ ]:
# trying a lil advance instructions now for stress testing
system_prompt = """/no_think You are a synthetic data generator.
Generate realistic synthetic data formatted exclusively as a Markdown table.
Do not include any explanations, introductory text, or concluding notes.
Start directly with the Markdown table header.
Strictly enforce all data types, categorical domains, and cross-column relational constraints."""

user_prompt = """Generate 6 realistic customer order records as a Markdown table.

Schema & Constraints:
- order_id: integer starting at 5001
- customer_name: string (include at least two non-English/international names, e.g., German or French)
- country: categorical (US, UK, DE, FR)
- shipping_address: string (realistic street address; must match the selected country)
- order_date: date in YYYY-MM-DD format (between 2024-01-01 and 2024-06-01)
- delivery_date: date in YYYY-MM-DD format (must be 2 to 7 days AFTER order_date)
- order_status: categorical ('Delivered', 'In Transit', 'Cancelled')
- notes: free text description that might contain commas or special characters (or 'None')

Relational Rule: If order_status is 'Cancelled', delivery_date must be empty/null."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
model_output = pipe(messages, max_new_tokens=30000)
assistant_response = model_output[0]["generated_text"][2]['content']
clean_response = assistant_response.split('</think>')[-1].strip()
display(Markdown(clean_response))

[transformers] Both `max_new_tokens` (=30000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


| order_id | customer_name       | country | shipping_address                          | order_date  | delivery_date | order_status     | notes                          |
|----------|---------------------|---------|-------------------------------------------|-------------|---------------|------------------|-------------------------------|
| 5001     | Müller, Anna        | DE      | Musterstraße 12, 12345 Berlin            | 2024-01-10  | 2024-01-15    | Delivered        | None                          |
| 5002     | Dupont, Jean        | FR      | 10 Rue des Écoles, 75001 Paris           | 2024-01-15  | 2024-01-20    | In Transit       | Shipped via DHL                |
| 5003     | Smith, John         | US      | 456 Main St, Springfield, IL 62704      | 2024-01-20  | 2024-01-25    | Delivered        | None                          |
| 5004     | Johnson, Sarah      | UK      | 789 London Road, Manchester M1 1AA      | 2024-02-01  | 2024-02-07    | Cancelled        | None                          |
| 5005     | O’Connor, Michael   | US      | 123 Oak Avenue, Chicago, IL 60611       | 2024-02-10  | 2024-02-16    | Delivered        | Gift for birthday             |
| 5006     | Wagner, Klaus       | DE      | 567 Berliner Straße, 10117 Berlin        | 2024-03-05  | 2024-03-12    | In Transit       | Urgent delivery required      |

In [ ]:
# since instructions were not met in the cancelled case, we revert back to csv, for bring more token efficient and stress test the model again
system_prompt = """/no_think You are a deterministic synthetic tabular data generator.
Output only valid CSV data without markdown formatting, backticks, or explanatory text.
Always output the exact CSV header first.
Follow every cross-column dependency strictly. If a condition requires a field to be null or empty, leave it as an empty value between commas (e.g., ",,")."""

user_prompt = """Generate 15 realistic customer order records as raw CSV.

Schema:
- order_id: sequential integer starting at 5001
- customer_name: string (diverse international names; wrap in quotes if containing commas)
- country: categorical (US, UK, DE, FR)
- shipping_address: string matching the country
- order_date: date in YYYY-MM-DD format (between 2024-01-01 and 2024-06-01)
- delivery_date: date in YYYY-MM-DD format (must be 2 to 7 days AFTER order_date)
- order_status: categorical ('Delivered', 'In Transit', 'Cancelled')
- notes: free text description (use 'None' if empty; wrap in quotes if it contains commas)

Relational Rules:
1. If order_status == 'Cancelled', delivery_date MUST be left blank (empty string).
2. If country == 'DE', customer_name must follow German naming conventions."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
model_output = pipe(messages, max_new_tokens=30000)
assistant_response = model_output[0]["generated_text"][2]['content']
clean_response = assistant_response.split('</think>')[-1].strip()
print(clean_response)

[transformers] Both `max_new_tokens` (=30000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"order_id","customer_name","country","shipping_address","order_date","delivery_date","order_status","notes"
5001,"Schmidt, Anna","DE","Musterstrasse 1, 80331 Munich, Germany","2024-01-05","2024-01-08","Delivered","None"
5002,"Smith, John","US","123 Main St, Springfield, IL, USA","2024-01-07","2024-01-10","In Transit","Shipped via FedEx"
5003,"Brown, Emily","UK","456 Oak Lane, Manchester, UK","2024-01-09","2024-01-11","Delivered","None"
5004,"Schmidt, Klaus","DE","Berliner Straße 45, 10178 Berlin, Germany","2024-01-10","2024-01-12","Cancelled","None"
5005,"Johnson, Michael","US","789 Pine Rd, Chicago, IL, USA","2024-01-12","2024-01-15","In Transit","Expected delivery tomorrow"
5006,"Williams, Sarah","UK","321 Maple Ave, London, UK","2024-01-14","2024-01-17","Delivered","None"
5007,"Müller, Lisa","DE","Rathenauer Straße 23, 10179 Berlin, Germany","2024-01-16","2024-01-19","In Transit","Shipped via DHL"
5008,"Taylor, David","US","876 Elm St, Dallas, TX, USA","2024-01-18","2024-01-21","Del

In [ ]:
# point here to be noted is that LLM is generating text token by token, left to right, therefore, it is generating delivery date first and order status later, leading to issues in the output, since delivery date is being added for cancelled orders also. Let's try if addition of thinking mitigates this issue
system_prompt = """You are a deterministic synthetic tabular data generator.
Output only valid CSV data without markdown formatting, backticks, or explanatory text.
Always output the exact CSV header first.
Follow every cross-column dependency strictly. If a condition requires a field to be null or empty, leave it as an empty value between commas (e.g., ",,")."""

user_prompt = """Generate 15 realistic customer order records as raw CSV.

Schema:
- order_id: sequential integer starting at 5001
- customer_name: string (diverse international names; wrap in quotes if containing commas)
- country: categorical (US, UK, DE, FR)
- shipping_address: string matching the country
- order_date: date in YYYY-MM-DD format (between 2024-01-01 and 2024-06-01)
- delivery_date: date in YYYY-MM-DD format (must be 2 to 7 days AFTER order_date)
- order_status: categorical ('Delivered', 'In Transit', 'Cancelled')
- notes: free text description (use 'None' if empty; wrap in quotes if it contains commas)

Relational Rules:
1. If order_status == 'Cancelled', delivery_date MUST be left blank (empty string).
2. If country == 'DE', customer_name must follow German naming conventions."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
model_output = pipe(messages, max_new_tokens=30000)
assistant_response = model_output[0]["generated_text"][2]['content']
clean_response = assistant_response.split('</think>')[-1].strip()
print(clean_response)

[transformers] Both `max_new_tokens` (=30000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


order_id,customer_name,country,shipping_address,order_date,delivery_date,order_status,notes
5001,"Doe, John",US,"123 Main St, Springfield, IL",2024-01-05,2024-01-07,Delivered,"None"
5002,"Brown, James",UK,"456 London Road, Manchester",2024-01-10,2024-01-12,In Transit,"Gift for birthday, wrapped in red"
5003,"Hans Müller",DE,"Königstraße 78, Berlin",2024-01-15,2024-01-18,Delivered,"None"
5004,"Sophie Martin",FR,"10 Rue de la Paix, Paris",2024-01-20,2024-01-23,In Transit,"Order contains fragile items"
5005,"Smith, Jane",US,"789 Pine Ave, Chicago, IL",2024-01-25,2024-01-27,Delivered,"None"
5006,"Davis, Emily",UK,"321 Oxford Street, London",2024-02-01,,Cancelled,"None"
5007,"Anna Schmidt",DE,"Münchner Straße 45, Munich",2024-02-05,2024-02-07,Delivered,"Gift for birthday"
5008,"Lucien Dubois",FR,"15 Avenue des Champs-Élysées, Paris",2024-02-10,2024-02-12,In Transit,"None"
5009,"Johnson, Michael",US,"654 Oak Street, New York, NY",2024-02-15,2024-02-17,Delivered,"None"
5010,"Wilson, Olivia",U

In [ ]:
# tradeoff in terms of thinking is time being consumed by the LLM. while earlier responses came in less than a minute, this one took 8 mins, this can also be due to the fact that in evening colab becomes slow, now even after using L4, this is becoming slow. though the system is working, lets now try this in gradio using the interface from pipeline and see if it works fine or not.
import gradio as gr

In [ ]:
gr.Interface.from_pipeline(pipe).launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9ae2611b31a15a825a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 410, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2330, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1690, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

In [ ]:
# this is not running because of issue at gradio's end, it is not yet compatible with current version of transformers, so we build a function that feeds into gradio and gets the output. First, lets look at what we need in this interface?

# we have a system prompt, that goes in automatically, user prompt gets added to it, which a user types in gradio. Once, this is done, we remove the user prompt from the input text box. Move that to a text box that can not be edited and is marked as Query.

# The system prompt + user prompt goes to the function that we create. runs through the LLM.

# Once this is done, the output is now passed on to another output box, which can not be edited and is shown as data. this completed one cycle. Then we have some buttons, such as copy: this copies the data to the clipboard, one button for refresh which wipes out the slate clean and gets ready for the next input. This is v1 for our capstone.

In [6]:
pipe = pipeline(
    "text-generation",
    model=MODEL,
    device_map='auto',
    dtype=torch.bfloat16,
    model_kwargs={"quantization_config": quant_config}
)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [7]:
system_prompt = """You are a deterministic synthetic tabular data generator.
Output only valid CSV data without markdown formatting, backticks, or explanatory text.
Always output the exact CSV header first.
Follow every cross-column dependency strictly. If a condition requires a field to be null or empty, leave it as an empty value between commas (e.g., ",,")."""

In [8]:
def generate_data(user_input):
  if user_input:
    messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_input}
    ]
    model_output = pipe(messages, max_new_tokens=30000)
    assistant_response = model_output[0]["generated_text"][2]['content']
    clean_response = assistant_response.split('</think>')[-1].strip()
    return clean_response
  else:
    return "No input given, please share details of data to be generated"

In [37]:
user_prompt = """Generate 2 realistic customer order records as raw CSV.

Schema:
- order_id: sequential integer starting at 5001
- customer_name: string (diverse international names; wrap in quotes if containing commas)
- country: categorical (US, UK, DE, FR)
- shipping_address: string matching the country
- order_date: date in YYYY-MM-DD format (between 2024-01-01 and 2024-06-01)
- delivery_date: date in YYYY-MM-DD format (must be 2 to 7 days AFTER order_date)
- order_status: categorical ('Delivered', 'In Transit', 'Cancelled')
- notes: free text description (use 'None' if empty; wrap in quotes if it contains commas)

Relational Rules:
1. If order_status == 'Cancelled', delivery_date MUST be left blank (empty string).
2. If country == 'DE', customer_name must follow German naming conventions."""

In [38]:
generate_data(user_prompt)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'"order_id","customer_name","country","shipping_address","order_date","delivery_date","order_status","notes"\n"5001","Anna Müller","DE","Kurfürstendamm 123, Berlin, 10117","2024-01-10","2024-01-13","Delivered","None"\n"5002","John Smith","US","123 Main St, Springfield, IL, 62704","2024-02-05","2024-02-08","In Transit","None"\n"5003","James Brown","UK","456 Oak Rd, Manchester, M1 2AB","2024-03-15","2024-03-18","Delivered","None"\n"5004","Lucie Dubois","FR","Rue de la Paix 45, Paris, 75001","2024-04-01","2024-04-04","In Transit","None"\n"5005","Emma Johnson","US","789 Pine St, Chicago, IL, 60611","2024-05-10","2024-05-12","Delivered","Shipped via UPS, tracking number 98765"\n"5006","Olivia Taylor","UK","321 Elm St, London, SW1A 1AA","2024-01-20","","Cancelled","None"\n"5007","Hans Schmidt","DE","Berliner Straße 45, Munich, 80331","2024-02-25","2024-03-01","Delivered","None"\n"5008","Pierre Martin","FR","Rue de Rivoli 78, Paris, 75001","2024-03-10","2024-03-12","In Transit","None"\n"5009"

In [9]:
import gradio as gr

In [ ]:
# adding copy function in the button itself,

def clear_info():
  return '', ''


with gr.Blocks() as demo:
  input_box = gr.Textbox(label="Enter Data Details")
  output_box = gr.Textbox(label="Generated Data", buttons=['copy'])
  with gr.Row():
    generate_btn = gr.Button("Generate data")
    clear_btn = gr.Button("Clear")

  generate_btn.click(fn=generate_data, inputs=input_box, outputs=output_box)
  clear_btn.click(fn=clear_info, outputs=[input_box, output_box])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2bf6415107c37b8e70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=30000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
